# Workflow Event Log Analysis

This notebook analyzes a fully synthetic warehouse event log for the Logistics Workflow & Robotics AI Field Practice Project.

The dataset is synthetic and does not contain real personal, company, resume, or training-material data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Works whether the notebook is run from the repository root or from /notebooks
DATA_PATH = Path('../data/synthetic_warehouse_event_log.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/synthetic_warehouse_event_log.csv')

df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['exception_type'] = df['exception_type'].fillna('')

df.head()

In [ ]:
summary = {
    'total_cases': df['case_id'].nunique(),
    'total_events': len(df),
    'activity_count': df['activity'].nunique(),
    'exception_events': int((df['exception_type'] != '').sum()),
    'rework_or_exception_events': int(df['status'].isin(['rework', 'exception']).sum()),
}

pd.Series(summary, name='value')

In [ ]:
activity_counts = df['activity'].value_counts().rename_axis('activity').reset_index(name='count')
activity_counts

In [ ]:
exception_counts = (
    df.loc[df['exception_type'] != '', 'exception_type']
    .value_counts()
    .rename_axis('exception_type')
    .reset_index(name='count')
)
exception_counts

In [ ]:
station_delay = (
    df.groupby('station')['delay_minutes']
    .agg(total_delay_minutes='sum', avg_delay_minutes='mean', event_count='count')
    .reset_index()
    .sort_values('total_delay_minutes', ascending=False)
)
station_delay

In [ ]:
case_durations = (
    df.groupby('case_id')['timestamp']
    .agg(case_start='min', case_end='max')
    .reset_index()
)
case_durations['case_duration_minutes'] = (
    (case_durations['case_end'] - case_durations['case_start']).dt.total_seconds() / 60
).round(1)

case_durations.describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(activity_counts['activity'], activity_counts['count'])
plt.title('Workflow Activity Frequency')
plt.xlabel('Activity')
plt.ylabel('Count')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(exception_counts['exception_type'], exception_counts['count'])
plt.title('Workflow Exception Frequency')
plt.xlabel('Exception type')
plt.ylabel('Count')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(case_durations['case_duration_minutes'], bins=20)
plt.title('Synthetic Case Duration Distribution')
plt.xlabel('Duration in minutes')
plt.ylabel('Number of cases')
plt.tight_layout()
plt.show()

## Interpretation guide

- `activity_counts` shows how often each workflow step appears in the synthetic event log.
- `exception_counts` shows which exception types occur most often.
- `station_delay` identifies stations with larger accumulated delays.
- `case_durations` estimates total process time for each synthetic order case.

These outputs can be used to discuss scanning, sorting, inspection, packing, loading support, exception review, and robotics/AI touchpoints.